In [5]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/ndaidcnd/Desktop/deidentification/deIdentification/')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task
from qc_package.scanner import DbScanner
import pandas as pd

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [6]:
TableDetailsModel.objects.filter(table_status=2).count()

289

In [7]:
qc_config = {
    "PATIENT_ID": {"prefix_value": "1001000", "length_of_value": 15},
    "ENCOUNTER_ID": {"prefix_value": "1001000", "length_of_value": 19},
}
db = DbDetailsModel.objects.get(db_name="deid_prod_1d")

db_scanner = DbScanner(
    db.source_db_config['connection_str'],
    db.destination_db_config['connection_str'],
    db.run_config["mapping_db_config"],
    qc_config
)

In [21]:
import os, json
from datetime import datetime

all_results = []

def is_phi_table(table_config):
    for col_conf in table_config['columns_details']:
        if col_conf['is_phi']:
            return True
    return False
    
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    # if table_obj.table_name in tables_for_qc:
    qc_result = table_obj.qc_result
    if not qc_result:
        continue
    one_table = {
        "autoqc_date": datetime.now().date(),
        "table_name": table_obj.table_name,
        "source_rows_count": qc_result['source_rows_count'],
        "dest_rows_count": qc_result['dest_rows_count'],
        "status": qc_result.get("final_qc_result", {}).get("is_qc_passed"),
        "ignore_rows_count": qc_result['ignore_rows_count'],
        "reason": qc_result.get("final_qc_result", {}).get("reason"),
        "is_phi": is_phi_table(table_obj.table_details_for_ui)
    }
    all_results.append(one_table)
len(all_results)

289

In [22]:
res_df = pd.DataFrame(all_results)
res_df.shape

(289, 8)

In [23]:
res_df['status'].value_counts()

status
True     215
False     74
Name: count, dtype: int64

In [ ]:
# res_df.to_csv("qc_results_staging_1d_12182025.csv", index=False)